# 🧠 EfficientNet-B0 기반 감정 분류 딥러닝 모델 설계, 학습 및 최종 검증

이 노트북은 전 세계적으로 검증받은 고성능 이미지 분류 알고리즘인 **EfficientNet-B0**를 활용하여 표정 이미지 감정 분류기를 학습하는 전체 과정을 교육용으로 담고 있습니다.

1단계부터 8단계까지 매끄럽게 흐르는 파이프라인을 경험하고, 최종적으로 학습 완료된 최적 가중치(`best_emotion_model.pth`)를 **불러와서(읽어와서)** 테스트 이미지에 대입해 실제 감정을 정확하게 분류하고 한글로 시각화해 봅니다.

## 🛠️ 1단계: 필수 라이브러리 설치 및 불러오기

파이토치와 시각화를 위한 맷플롯립(Matplotlib) 등을 설치하고 정상 로드되는지 테스트합니다.

In [ ]:
# 가상환경 내에 필수적인 인공지능 라이브러리들을 설치합니다.
# (최초 실행 후 다음 실행부터는 이 셀을 실행하지 않으셔도 괜찮습니다)
%pip install torch torchvision matplotlib pillow

In [ ]:
# 필요한 라이브러리를 안전하게 불러옵니다.
import os
import torch
import torch.nn as nn
from torchvision import models

print(f"🔥 PyTorch 버전: {torch.__version__}")
print("✨ 딥러닝 핵심 엔진 모듈이 오류 없이 성공적으로 임포트되었습니다.")

## 🧬 2단계: EfficientNet-B0 사전 학습 모델 로드 및 출력층 개조

**전이 학습(Transfer Learning)** 기법을 사용하여, 이미 수백만 장의 이미지 데이터셋(ImageNet)을 사전에 깊게 학습하여 이미지의 특징을 훌륭하게 잡아내는 특징 추출기(Feature Extractor)의 지식은 그대로 재사용하고, 맨 뒷단에 붙어 있는 분류 레이어(Classifier)만 우리의 목표 감정 개수(7개)에 맞게 개조합니다.

In [ ]:
def get_emotion_efficientnet(num_classes=7):
    """
    사전 학습된 EfficientNet-B0를 불러와 우리의 감정 분류 개수(7개)에 맞춰 마지막 선형 레이어를 수정하는 함수입니다.
    """
    print("🚀 EfficientNet-B0 사전 학습 가중치를 다운로드 및 로드하는 중...")
    
    # 1. 완성도 높게 사전 학습된 기본 가중치를 불러옵니다.
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    
    # 2. EfficientNet-B0 분류기 구조: [0] Dropout, [1] Linear 계층으로 구성되어 있습니다.
    # 마지막 출력부 직전의 특징 차원 크기(1280차원)를 추출합니다.
    in_features = model.classifier[1].in_features
    
    # 3. 기존의 1000개 클래스용 선형 계층을 우리의 감정 개수(7개)에 맞춘 새로운 선형 계층으로 정교하게 교체합니다.
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    return model

## 🧪 3단계: 가짜(Dummy) 데이터를 이용한 모델 작동성 테스트

실제 이미지를 넣기 전에 임의의 형태를 가진 가짜 이미지 텐서(Tensor)를 주입해 보아 계산 형태가 어긋나지 않는지 미리 진단합니다.

In [ ]:
# 감정 클래스 개수 정의 (기쁨, 슬픔, 화남, 놀람, 혐오, 공포, 무표정 = 7개)
num_emotions = 7
emotion_model = get_emotion_efficientnet(num_classes=num_emotions)

# EfficientNet은 RGB 3채널의 224x224 해상도 이미지를 표준 규격으로 받아들입니다.
# [배치 크기(1장), 채널 수(3), 이미지 높이(224), 이미지 너비(224)]
dummy_image = torch.randn(1, 3, 224, 224)

# 모델에 주입하여 예측(순전파 연산)을 돌려봅니다.
output = emotion_model(dummy_image)

print("\n================ 📊 모델 가상 추론 테스트 결과 ================")
print("✅ 딥러닝 신경망 조립 및 순전파 출력이 아주 안정적으로 작동합니다!")
print(f"- 📥 입력 데이터 형태 (Input Shape)  : {dummy_image.shape}  -> [배치크기, 채널, 높이, 너비]")
print(f"- 📤 출력 데이터 형태 (Output Shape) : {output.shape}   -> [배치크기, 감정 클래스 개수]")
print("===========================================================")

## 📂 4단계: PyTorch Dataset 및 DataLoader 구축

앞선 단계에서 8:1:1로 깔끔하게 물리 분할을 완료한 실제 데이터셋 폴더(`data/train`, `data/val`)를 연동합니다. 
학습용 이미지에는 과적합(Overfitting)을 철저히 차단하기 위해 **데이터 증강(Data Augmentation)** 기법(좌우 대칭 반전, 회전 등)을 부가하여 딥러닝 학습 효과를 고도화합니다.

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 분할된 데이터셋의 저장 폴더 경로를 확인합니다.
train_dir = 'data/train'
val_dir = 'data/val'

# 1. 데이터셋 전처리(Transforms) 정책 수립 (오류 원천 수정)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),               # EfficientNet 입력 최적 크기
    transforms.RandomHorizontalFlip(p=0.5),      # 좌우 반전 적용
    transforms.RandomRotation(degrees=10),       # 가벼운 회전 왜곡
    transforms.ToTensor(),                       # 이미지를 수학적 텐서(Tensor)로 변환
    transforms.Normalize(                        # ImageNet 표준화 기준 정규화
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])  # 👈 대괄호와 소괄호를 완벽하게 닫아 구문 오류를 차단했습니다!

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),               # 검증 데이터는 증강 없이 해상도 조절만 수행
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])  # 👈 대괄호와 소괄호를 완벽하게 닫아 구문 오류를 차단했습니다!

# 2. ImageFolder를 활용해 감정 라벨별 폴더 자동 매핑 및 데이터셋 적재
if not os.path.exists(train_dir):
    print(f"⚠️ 경고: '{train_dir}' 폴더가 없습니다. dataset_splitter를 먼저 작동시켜주세요.")
else:
    train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
    val_dataset = ImageFolder(root=val_dir, transform=val_transforms)

    # 3. 미니배치 단위로 빠르게 공급할 DataLoader 설정 (배치 32 지정)
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    print("=== 📂 데이터셋 로딩 및 미니배치 구성 완료 ===")
    print(f"- 🏋️ 학습용(Train) 이미지 개수  : {len(train_dataset):,}장 (배치 묶음: {len(train_loader)}개)")
    print(f"- 🧪 검증용(Val) 이미지 개수    : {len(val_dataset):,}장 (배치 묶음: {len(val_loader)}개)")
    print(f"- 🏷️ 감정 클래스 이름 매핑 정보 : {train_dataset.class_to_idx}")

## ⚡ 5단계: GPU 가속 설정 및 오차/최적화 함수 세팅

학습 속도를 기하급수적으로 끌어올리기 위해 컴퓨터 내의 그래픽카드 장치(CUDA GPU)를 자동으로 인식하도록 연동하고, AI 학습의 핵심인 **오차 측정 도구(Loss Function)**와 **가중치 업데이트 도구(Optimizer)**를 지정합니다.

In [ ]:
import torch.optim as optim

# 1. NVIDIA GPU(CUDA) 존재 시 자동 가속, 없으면 CPU로 자동 백업 작동
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 현재 인공지능 가속에 할당된 딥러닝 연산 디바이스: {device}")

# 2. 설계된 감정 인식 모델을 GPU(혹은 CPU) 장치 메모리로 번개같이 이식
emotion_model.to(device)

# 3. 다중 분류 학습에 가장 탁월한 교차 엔트로피 오차 손실함수(CrossEntropyLoss) 선언
criterion = nn.CrossEntropyLoss()

# 4. 미세 조정(Fine-tuning)을 최적으로 보조하는 AdamW 최적화 기법 도입
# 기존 사전 학습 가중치가 파괴되는 것을 예방하기 위해 매우 촘촘하고 세밀한 갱신폭(Learning Rate = 1e-4)을 지정합니다.
optimizer = optim.AdamW(emotion_model.parameters(), lr=1e-4, weight_decay=1e-2)

print("✅ 오차 손실 연산식, 가중치 업데이트용 Optimizer, 연산 하드웨어 셋업 완료!")

## 🔄 6단계: 실전 딥러닝 학습 및 검증 루프 (Epoch 연산)

본격적으로 모든 데이터를 훑는 에포크(Epoch) 단위 학습을 개시합니다. 
매 회차가 끝날 때마다 학습 상태를 확인하고, 특히 **검증 정확도(Val Accuracy)가 역대 최고를 달성한 시점**의 영특한 가중치만을 선별하여 컴퓨터 파일(`best_emotion_model.pth`)로 소중하게 영구 저장합니다.

In [ ]:
import time

# 훈련의 반복 횟수를 가볍게 3회로 지정해 훈련이 정상 작동하는지 테스트합니다.
num_epochs = 3
best_val_acc = 0.0  # 최고 성능 돌파 기준이 되는 변수 기록용

print("🏁 감정 표정 분류 딥러닝 학습 세션을 시작합니다...\n")

for epoch in range(num_epochs):
    start_time = time.time()
    
    # ==========================================
    # [🏋️ 훈련 상태 활성화 - Training Phase]
    # ==========================================
    emotion_model.train()  # 드롭아웃 및 배치 정규화 등 학습용 활성 스위치 온
    train_loss = 0.0
    train_corrects = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # 매 미니배치마다 직전 미니배치에서 쌓인 그라디언트 초기화
        optimizer.zero_grad()
        
        # 1) 순전파 (Forward Pass)
        outputs = emotion_model(inputs)
        loss = criterion(outputs, labels)
        
        # 2) 역전파 (Backward Pass)
        loss.backward()
        
        # 3) 가중치 수정 (Weight Update)
        optimizer.step()
        
        # 통계 계산을 위한 오차 및 정답 수치 적립
        _, preds = torch.max(outputs, 1)
        train_loss += loss.item() * inputs.size(0)
        train_corrects += torch.sum(preds == labels.data)
        
    # 1 Epoch 평균 손실값 및 평균 정확도 산출
    epoch_train_loss = train_loss / len(train_dataset)
    epoch_train_acc = train_corrects.double() / len(train_dataset)
    
    # ==========================================
    # [🧪 검증 평가 상태 활성화 - Validation Phase]
    # ==========================================
    emotion_model.eval()  # 예측 성능 측정이 목적이므로 드롭아웃 등 비활성화
    val_loss = 0.0
    val_corrects = 0
    
    # 검증 시에는 그라디언트 계산 일체 중단
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = emotion_model(inputs)
            loss = criterion(outputs, labels)
            
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    # 1 Epoch 검증용 평균 손실값 및 평균 정확도 산출
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_corrects.double() / len(val_dataset)
    
    epoch_time = time.time() - start_time
    
    print(f"🔄 [{epoch+1}/{num_epochs} Epoch] 훈련 진행 시간: {epoch_time:.1f}초")
    print(f"  - 🏋️ 학습 데이터  -> Loss: {epoch_train_loss:.4f} | Accuracy: {epoch_train_acc:.4f}")
    print(f"  - 🧪 검증 데이터  -> Loss: {epoch_val_loss:.4f} | Accuracy: {epoch_val_acc:.4f}")
    
    # ==========================================
    # [🌟 역사상 가장 똑똑한 베스트 시점의 가중치 영구 저장]
    # ==========================================
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        # 가장 현명해진 가중치 데이터를 디스크에 저장합니다.
        torch.save(emotion_model.state_dict(), 'best_emotion_model.pth')
        print(f"  🌟 [발견] 역대 최고의 검증 성능 갱신 돌파! 최적 가중치('best_emotion_model.pth') 파일로 보존 완료.")
    print("-" * 70)

print(f"🎉 학습 종결! 최고 검증 정확도: {best_val_acc:.4f}")

## 💾 7단계: 저장된 최적 모델 가중치 읽기(Load) 및 테스트 데이터 최종 검증

학습이 잘 마무리되어 파일(`best_emotion_model.pth`)로 생성된 최적 가중치 모델을 **처음부터 다시 읽어서(Load)**, 단 한 번도 학습이나 검증 과정에 개입한 적이 없는 완전히 독립된 **테스트 데이터셋(`data/test`)**에 대입하여 객관적인 최종 실력을 리포트합니다.

In [ ]:
# 1. 테스트 전용 전처리(Transforms) 설정
# 학습 때의 이미지 왜곡(증강)을 일체 해제하고 크기 조정과 표준화 정규화만 수행합니다.
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])  # 👈 대괄호와 소괄호 닫기 완벽 반영!

# 2. 테스트 이미지 폴더 불러오기
test_dir = 'data/test'
if not os.path.exists(test_dir):
    print(f"⚠️ 오류: '{test_dir}' 경로를 찾을 수 없습니다. 데이터가 올바르게 분할되어 있는지 확인해 주세요.")
else:
    test_dataset = ImageFolder(root=test_dir, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
    print("=== 📂 테스트 전용 데이터셋 연동 성공 ===")
    print(f"- 🧪 실전 테스트용 이미지 개수: {len(test_dataset):,}장 (배치 묶음: {len(test_loader)}개)")
    
    # 3. 저장된 최적의 모델 가중치 파일 읽어오기 (Load)
    evaluation_model = get_emotion_efficientnet(num_classes=num_emotions)
    
    model_save_path = 'best_emotion_model.pth'
    if not os.path.exists(model_save_path):
        print(f"⚠️ 오류: '{model_save_path}' 저장된 가중치 파일이 없습니다. 6단계 학습을 먼저 수행해 주세요.")
    else: 
        # 저장된 가중치 딕셔너리를 읽어서 새로 만든 모델에 완전히 이식합니다.
        evaluation_model.load_state_dict(torch.load(model_save_path, map_location=device))
        evaluation_model.to(device)
        print(f"✅ 최적 가중치 파일 '{model_save_path}' 로딩(읽기) 완료!")
        
        # 4. 실전 테스트 데이터 검증 연산 시작
        evaluation_model.eval()
        total_test_loss = 0.0
        total_test_corrects = 0
        
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = evaluation_model(inputs)
                loss = criterion(outputs, labels)
                
                _, preds = torch.max(outputs, 1)
                total_test_loss += loss.item() * inputs.size(0)
                total_test_corrects += torch.sum(preds == labels.data)
                
        final_test_loss = total_test_loss / len(test_dataset)
        final_test_acc = total_test_corrects.double() / len(test_dataset)
        
        print("\n================ 📊 최종 실전 테스트 세트 평가 리포트 ================")
        print(f"- 📉 실전 테스트 평균 손실값(Test Loss)     : {final_test_loss:.4f}")
        print(f"- 📈 실전 테스트 최종 정확도(Test Accuracy) : {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")
        print("=======================================================================")

## 🖼️ 8단계: 테스트 데이터 무작위 추출 및 감정 예측 시각화

인공지능의 실전 예측 결과를 우리 눈으로 직접 확인해 봅니다. 
테스트 데이터셋에서 무작위로 **5장의 표정 이미지**를 선택하여 모델에 입히고, 인공지능이 도출한 예측값과 실제 데이터에 등록된 정답(Label)을 비교하여 한글로 예쁘게 사진 위에 시각화합니다.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Windows OS 기반 한글 맑은 고딕 세팅
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}
random_indices = random.sample(range(len(test_dataset)), 5)

plt.figure(figsize=(16, 5))

for count, index in enumerate(random_indices):
    img_tensor, label = test_dataset[index]
    input_tensor = img_tensor.unsqueeze(0).to(device)
    
    evaluation_model.eval()
    with torch.no_grad():
        outputs = evaluation_model(input_tensor)
        _, pred = torch.max(outputs, 1)
        pred_index = pred.item()
        
    true_emotion = idx_to_class[label]
    predicted_emotion = idx_to_class[pred_index]
    
    # 역정규화 복원
    img_numpy = img_tensor.numpy().transpose((1, 2, 0))
    mean_values = np.array([0.485, 0.456, 0.406])
    std_values = np.array([0.229, 0.224, 0.225])
    img_numpy = std_values * img_numpy + mean_values
    img_numpy = np.clip(img_numpy, 0, 1)
    
    plt.subplot(1, 5, count + 1)
    plt.imshow(img_numpy)
    
    if pred_index == label:
        text_color = 'green'
        match_status = '일치 ✅'
    else:
        text_color = 'red'
        match_status = '불일치 ❌'
        
    plt.title(f"실제: {true_emotion}\n예측: {predicted_emotion}\n({match_status})", color=text_color, fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()